# EcoPulse AI — Air Quality Exploratory Data Analysis
Historical daily air-quality observations for Indian cities (2015–2020).
Run the cells from top to bottom (**Run All**). The notebook reads the cleaned file created by `src/preprocess_air.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False})
TEAL, DARK, MINT = "#287D81", "#104B47", "#76B5A3"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data" / "processed" / "air_pm25.csv", parse_dates=["Date"])
df.head()

## 1. Dataset overview

In [ ]:
print("Rows:", len(df), "| Cities:", df["City"].nunique())
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
missing = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
missing[missing > 0]

## 2. How is PM2.5 distributed?
The red line is India's 24-hour limit (60 µg/m³); green is the WHO 2021 guideline (15 µg/m³).

In [ ]:
ax = df["PM2.5"].clip(upper=400).plot.hist(bins=60, color=TEAL, edgecolor="white")
ax.axvline(60, color="#D9453D", ls="--", label="India limit (60)")
ax.axvline(15, color="#1E9E5A", ls=":", label="WHO guideline (15)")
ax.set_xlabel("PM2.5 (µg/m³), values above 400 grouped at 400"); ax.set_ylabel("Days"); ax.legend()
plt.show()

## 3. Which cities are most polluted?

In [ ]:
city_avg = df.groupby("City")["PM2.5"].mean().sort_values(ascending=False)
colors = ["#7A1F2B" if v > 120 else "#D9453D" if v > 90 else "#EE8A1F" if v > 60 else "#7CB82F" for v in city_avg]
ax = city_avg.plot.bar(color=colors, figsize=(11, 4.5))
ax.axhline(60, color="#D9453D", ls="--"); ax.set_ylabel("Average PM2.5 (µg/m³)")
plt.xticks(rotation=60, ha="right"); plt.show()
city_avg.round(1).head(5)

## 4. Seasonal pattern
Pollution usually peaks in winter (Nov–Jan) and drops during the monsoon.

In [ ]:
monthly = df.groupby(df["Date"].dt.month)["PM2.5"].mean()
ax = monthly.plot(marker="o", color=DARK, lw=3)
ax.set_xticks(range(1, 13)); ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_ylabel("Average PM2.5 (µg/m³)"); plt.show()

## 5. How do pollutants relate to each other?

In [ ]:
cols = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3"]
corr = df[cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="BrBG", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, shrink=0.8); plt.show()

## 6. Model comparison
Results saved by `src/evaluate_air_models.py` (chronological 80/20 split). Higher R² and lower MAE/RMSE are better.

In [ ]:
results = pd.read_csv(ROOT / "reports" / "air_model_comparison.csv")
display(results)
ax = results.set_index("Algorithm")["R2"].plot.barh(color=[MINT]*4 + [DARK])
ax.set_xlabel("R² score"); plt.show()

## Summary
* Delhi and a few northern/eastern cities show the highest average PM2.5.
* PM2.5 follows a clear seasonal cycle with winter peaks.
* Random Forest gives the best test performance among the five regression models.
* These models estimate PM2.5 from supplied pollutant readings; they do not forecast the future.